In [ ]:

# from pathlib import Path
# from typing import List, Tuple, Dict
# import json, random, itertools, statistics as st

# import numpy as np
# import pandas as pd
# from datasets import Dataset, DatasetDict
# from sklearn.model_selection import KFold, train_test_split
# from collections import Counter
# 
# from sklearn.neighbors import BallTree

# import torch
# from transformers import (
#     AutoTokenizer,
#     DataCollatorForTokenClassification,
#     AutoModelForTokenClassification,
#     TrainingArguments,
#     Trainer,
# )

# from seqeval.metrics import f1_score, classification_report
# from tqdm.auto import tqdm

In [2]:
from datasets import DatasetDict, Dataset
import random
import os, re
import openai

import numpy as np
import pandas as pd
import json, random, itertools, statistics as st

from sklearn.metrics import f1_score
from dotenv import load_dotenv
from typing import List, Tuple, Dict
from sklearn.model_selection import KFold, train_test_split
from collections import Counter
from sentence_transformers import SentenceTransformer
from seqeval.metrics import classification_report, f1_score as ner_f1
from IPython.display import clear_output


c:\Users\user\Documents\mestrado\ner_splits\ner_splits\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
load_dotenv() 
openai.api_key = os.getenv("OPENAI_API_KEY")

In [4]:
client = openai.OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

# Configuração e Verificação Inicial

In [5]:
JSON_PATH = "data/geocorpus-v2.json"        # ajuste se estiver noutra pasta
SEED_GLOBAL = 42
FEW_SHOT_K   = 50
MODEL_NAME   = "o4-mini"
random.seed(SEED_GLOBAL)
np.random.seed(SEED_GLOBAL)

# ---------- ler o arquivo ----------
with open(JSON_PATH, encoding="utf-8") as f:
    raw = json.load(f)

# cada entrada já tem tokens + ner_tokens

In [6]:
records_geo = [
    {
        "sentence_id": i,
        "tokens"     : item["tokens"],
        "ner_tags"   : item["ner_tokens"],
    }
    for i, item in enumerate(raw)
]

geocorpus_full = Dataset.from_list(records_geo)

In [7]:
# lista de rótulos (ordem alfabética garante consistência entre runs)
label_list = sorted({l for sent in geocorpus_full["ner_tags"] for l in sent})
label2id   = {l: i for i, l in enumerate(label_list)}
id2label   = {i: l for l, i in label2id.items()}
NUM_LABELS = len(label_list)

# Splits

In [8]:
# --- hold-out 80/20 (standard para este corpus) -----------------
standard_geo = geocorpus_full.train_test_split(
    test_size=0.2, seed=SEED_GLOBAL
)

# --- 30 divisões aleatórias -------------------------------------
def random_splits(
    ds: Dataset, test_size=0.2, seeds: List[int] = range(30)
) -> List[DatasetDict]:
    triples = []
    for s in seeds:
        train, dev = ds.train_test_split(test_size=test_size, seed=s).values()
        triples.append(DatasetDict(train=train, dev=dev))
    return triples

# --- heurística comprimento (20 % piores casos) -----------------
def split_heur_length(ds, top_pct=0.20):
    lengths   = np.array([len(t) for t in ds["tokens"]])
    thr       = np.percentile(lengths, 100*(1-top_pct))
    idx_long  = np.where(lengths >= thr)[0]
    idx_short = np.where(lengths <  thr)[0]
    return DatasetDict(
        train=ds.select(idx_short.tolist()),
        dev  =ds.select(idx_long.tolist())
    )

# --- heurística raridade ----------------------------------------
def split_heur_rare(ds, freq_thr=5):
    freq = {}
    for sent in ds["tokens"]:
        for w in sent:
            freq[w.lower()] = freq.get(w.lower(), 0) + 1
    rare = {w for w,c in freq.items() if c <= freq_thr}
    keep_dev = [any(w.lower() in rare for w in sent) for sent in ds["tokens"]]
    idx_dev   = [i for i,b in enumerate(keep_dev) if b]
    idx_train = [i for i,b in enumerate(keep_dev) if not b]
    return DatasetDict(train=ds.select(idx_train), dev=ds.select(idx_dev))

# --- adversarial (versão rápida, 10 % do corpus) ----------------
def split_adversarial(ds: Dataset, pct_test: float = 0.20) -> DatasetDict:
    k = int(len(ds) * pct_test)
    model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")
    embeds = model.encode([" ".join(t) for t in ds["tokens"]], show_progress_bar=False)
    tree = BallTree(embeds, leaf_size=40)

    idx_train, idx_test = set(range(len(ds))), []
    # semente = ponto mais central
    seed_idx = np.argmax(np.linalg.norm(embeds - embeds.mean(0), axis=1))
    idx_train.remove(seed_idx)
    idx_test.append(seed_idx)
    print(k)
    while len(idx_test) < k:
        print(len(idx_test))
        dists, _ = tree.query(embeds[list(idx_train)], k=1, return_distance=True)
        nxt = list(idx_train)[int(np.argmax(dists))]
        idx_train.remove(nxt)
        idx_test.append(nxt)

    return DatasetDict(
        train=ds.select(sorted(idx_train)),
        dev=ds.select(sorted(idx_test)),
    )

    # Maximizando Wassertein Distance


def split_adversarial_fast(ds: Dataset, pct_test: float = 0.20) -> DatasetDict:
    """
    Farthest-Point Sampling aproximando Wasserstein – versão vetorizada.
    Seleciona pct_test (~20 %) das sentenças como conjunto 'dev'.
    """
    k = int(len(ds) * pct_test)
    model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

    embeds = model.encode(
        [" ".join(t) for t in ds["tokens"]],
        show_progress_bar=True,
        convert_to_numpy=True,
        normalize_embeddings=True,  # acelera distância euclidiana ≈ cos
    )

    n = embeds.shape[0]
    idx_all = np.arange(n)

    # 1) ponto mais "central" (norma mais distante da média)
    seed_idx = np.argmax(np.linalg.norm(embeds - embeds.mean(0), axis=1))
    selected = [seed_idx]

    # 2) vetor de distâncias mínimas a qualquer ponto já escolhido
    min_dists = np.linalg.norm(embeds - embeds[seed_idx], axis=1)

    while len(selected) < k:
        next_idx = np.argmax(min_dists)
        selected.append(next_idx)

        # atualiza min_dists com a distância ao novo ponto — tudo de uma vez
        d_new = np.linalg.norm(embeds - embeds[next_idx], axis=1)
        min_dists = np.minimum(min_dists, d_new)

    train_idx = np.setdiff1d(idx_all, selected, assume_unique=True)

    return DatasetDict(
        train=ds.select(train_idx.tolist()),
        dev=ds.select(selected),
    )

In [9]:
geocorpus_full

Dataset({
    features: ['sentence_id', 'tokens', 'ner_tags'],
    num_rows: 5272
})

In [10]:
random_geo = random_splits(geocorpus_full, test_size=0.05)
heur_len_geo = split_heur_length(geocorpus_full, top_pct=0.05)
heur_rare_geo = split_heur_rare(geocorpus_full, freq_thr=5)
advers_geo = split_adversarial_fast(geocorpus_full, pct_test=0.05)

Batches: 100%|██████████| 165/165 [00:30<00:00,  5.41it/s]


# Tokenização e Métricas

In [11]:
def tokens_to_text(tokens: list[str]) -> str:
    return " ".join(tokens) 

In [12]:
def select_few_shot(exemplars: Dataset, k=FEW_SHOT_K) -> list[dict]:
    # amostra k exemplos aleatórios (pode trocar por stratified sampling)
    return random.sample(list(exemplars), k)

In [13]:
def fmt_example(ex):
    tokens = ex["tokens"]
    tags   = ex["ner_tags"]

    # 1) sentença “crua”
    sentence = " ".join(tokens)

    # 2) linha de tags no estilo original
    tags_line = " ".join(tags)

    # 3) pares enumerados: Token n -> TAG-token
    pairs = [
        f"{i+1}-{tag}"
        for i, (tag) in enumerate(tags)
    ]
    pairs_line = " ".join(pairs)

    txt = tokens_to_text(ex["tokens"])


    return f"Tokens: {txt}\nTags:   {pairs_line}\n\n"

In [14]:
def build_prompt(few_shot: list[dict], query_tokens: list[str]) -> str:
    prompt = """Você é um analista especialista em NER. Sua tarefa é rotular cada token da sentença abaixo usando o esquema IOB2.  
            Siga **exatamente** as instruções:

            • Use **apenas** as seguintes etiquetas (distingue maiúsculas-minúsculas):  
            O B-baciaSedimentar I-baciaSedimentar B-epoca I-epoca B-idade I-idade B-periodo I-periodo B-eon I-eon B-era I-era B-magmaticas I-magmaticas B-metamorficas I-metamorficas B-sedimentaresSiliciclasticas I-sedimentaresSiliciclasticas B-sedimentaresCarbonaticas I-sedimentaresCarbonaticas B-unidadeEstratigrafica I-unidadeEstratigrafica B-contextoGeologicoDeBacia I-contextoGeologicoDeBacia B-ambienteSedimentacao I-ambienteSedimentacao B-constituinteRochaSedimentar I-constituinteRochaSedimentar B-fosseis I-fosseis B-planctonico I-planctonico B-bentonico I-bentonico B-mineral I-mineral B-procedimentoMetodologico I-procedimentoMetodologico

            • Produza **uma sequência de etiquetas separadas por espaço**, na mesma ordem e quantidade de tokens fornecidos.  
            • Não inclua nenhum texto extra (títulos, comentários, pontuação ou quebras de linha), apenas a posição do token e a sua etiqueta.  
            • Se não houver entidade para um token, use “O”.
            
            Exemplos de classificação: \n """
    
    for ex in few_shot:
        prompt += fmt_example(ex) + "\n"

    tamanho = len(query_tokens)
    # agora a instância a ser anotada
    prompt += f""": Agora, classifique os comentários abaixo seguindo o mesmo raciocínio, devolva o número do token e a etiqueta correspondente, separados por hífen.: 
                    Tokens: {tokens_to_text(query_tokens)}
                    Tags:
                    Assim temos {tamanho} tokens, e você deve produzir {tamanho} etiquetas separadas por espaço, seguindo o esquema IOB2."""
    return prompt

In [15]:
def classify_with_chatgpt(prompt: str) -> list[str]:
    is_o_series = bool(re.match(r"o\d", MODEL_NAME))

    kwargs = {
        "model": MODEL_NAME,
        "messages": [{"role": "user", "content": prompt}],
        # só envia se não for o-series
        "temperature": openai.NOT_GIVEN if is_o_series else 0.0,
        # nome correto do parâmetro dependendo da família
        ("max_completion_tokens" if is_o_series else "max_tokens"): len(prompt.split())*2 ,
    }

    print(prompt)

    # resp = client.chat.completions.create(
    #     model=MODEL_NAME,
    #     messages=[{"role":"user","content":prompt}],
    #     "temperature": NOT_GIVEN if is_o_series else 0.0,
    #     max_completion_tokens
    #     =len(prompt.split())*2  # suficiente p/ a sequência de tags
    # )

    resp = client.chat.completions.create(**kwargs)
    return resp.choices[0].message.content.strip().split()

In [16]:
def compute_metrics(p):
    logits, labels = p
    preds = np.argmax(logits, -1)
    out_pred, out_true = [], []
    for p_i, l_i in zip(preds, labels):
        mask = l_i != -100
        out_pred.append([id2label[idx] for idx in p_i[mask]])
        out_true.append([id2label[idx] for idx in l_i[mask]])
    return {"f1": f1_score(out_true, out_pred)}

In [17]:
def run_experiment_ner(split, seed):
    random.seed(seed)
    golds, preds = [], []
    for ex in split["dev"]:
        few = select_few_shot(split["train"], FEW_SHOT_K)
        prompt = build_prompt(few, ex["tokens"])
        pred_tags = classify_with_chatgpt(prompt)

        golds.append(ex["ner_tags"])
        preds.append(pred_tags)

    print(classification_report(golds, preds))             # relatório completo
    print("F1-macro NER:", ner_f1(golds, preds, average="macro"))

In [18]:
split_base = heur_len_geo

In [19]:
max_len_split = len(split_base["dev"])

In [20]:
split = heur_len_geo

In [21]:
atual_len_split = len(split["dev"])

In [22]:
dev = split["dev"] 
if max_len_split < atual_len_split:
    dev_ajustado = dev.select(range(max_len_split))
else:
    dev_ajustado = dev

In [23]:
total = len(dev_ajustado)
print(total)   

279


In [24]:
random.seed(SEED_GLOBAL)
golds, preds = [], []
realizado = 0
few = select_few_shot(split["train"], FEW_SHOT_K)


for ex in dev_ajustado:
        print(ex["tokens"])
        prompt = build_prompt(few, ex["tokens"])
        pred_tags = classify_with_chatgpt(prompt)
        golds.append(ex["ner_tags"])
        preds.append(pred_tags)
        realizado += 1
        clear_output(wait=True)
        print(f'Total de exemplos processados:', {realizado/total})
        

Total de exemplos processados: {1.0}


In [25]:
new_preds = []
for pred in preds:
    ajuste_preds = []
    for pred_token in pred:
        _, pred_ajustado = pred_token.split('-', 1)
        ajuste_preds.append(pred_ajustado)

        
    new_preds.append(ajuste_preds)

In [26]:
[len(gold) for gold in golds]

[133,
 82,
 78,
 65,
 82,
 74,
 70,
 77,
 70,
 66,
 90,
 73,
 80,
 65,
 63,
 63,
 95,
 91,
 99,
 77,
 88,
 112,
 101,
 64,
 74,
 63,
 81,
 107,
 81,
 90,
 72,
 64,
 78,
 74,
 66,
 89,
 65,
 64,
 83,
 84,
 78,
 70,
 67,
 82,
 92,
 109,
 69,
 103,
 127,
 115,
 77,
 92,
 78,
 89,
 65,
 102,
 85,
 77,
 71,
 70,
 63,
 75,
 83,
 79,
 66,
 103,
 64,
 82,
 74,
 66,
 84,
 65,
 69,
 83,
 64,
 85,
 91,
 193,
 110,
 78,
 66,
 64,
 63,
 95,
 66,
 78,
 80,
 85,
 69,
 93,
 124,
 71,
 64,
 99,
 116,
 86,
 128,
 63,
 92,
 79,
 75,
 65,
 76,
 101,
 63,
 146,
 77,
 68,
 63,
 165,
 68,
 66,
 70,
 102,
 80,
 64,
 238,
 91,
 66,
 83,
 68,
 63,
 64,
 93,
 63,
 113,
 122,
 63,
 84,
 91,
 69,
 89,
 74,
 79,
 110,
 79,
 104,
 81,
 72,
 103,
 105,
 103,
 75,
 68,
 65,
 66,
 66,
 94,
 78,
 82,
 69,
 63,
 66,
 69,
 93,
 69,
 104,
 68,
 63,
 81,
 81,
 155,
 71,
 93,
 71,
 66,
 66,
 70,
 74,
 124,
 85,
 171,
 130,
 102,
 95,
 184,
 88,
 99,
 79,
 65,
 91,
 99,
 138,
 85,
 73,
 68,
 74,
 82,
 70,
 96,
 82,
 86,
 85,


In [27]:
[len(pred) for pred in preds]

[133,
 82,
 78,
 65,
 82,
 74,
 70,
 77,
 70,
 66,
 90,
 73,
 80,
 65,
 63,
 63,
 95,
 91,
 99,
 77,
 88,
 112,
 101,
 64,
 74,
 63,
 81,
 107,
 81,
 90,
 72,
 64,
 78,
 74,
 66,
 89,
 65,
 64,
 83,
 84,
 78,
 70,
 67,
 82,
 92,
 109,
 69,
 103,
 129,
 115,
 77,
 92,
 78,
 89,
 65,
 102,
 85,
 77,
 71,
 70,
 63,
 75,
 83,
 79,
 66,
 103,
 64,
 82,
 74,
 66,
 84,
 65,
 69,
 83,
 64,
 85,
 91,
 193,
 108,
 78,
 66,
 64,
 63,
 97,
 66,
 78,
 80,
 85,
 69,
 93,
 124,
 71,
 64,
 99,
 116,
 86,
 128,
 63,
 92,
 79,
 75,
 65,
 76,
 101,
 63,
 146,
 77,
 68,
 63,
 165,
 68,
 66,
 70,
 102,
 80,
 64,
 0,
 91,
 66,
 83,
 68,
 63,
 64,
 93,
 63,
 113,
 122,
 63,
 84,
 91,
 69,
 89,
 74,
 79,
 110,
 79,
 104,
 81,
 72,
 103,
 105,
 103,
 75,
 68,
 65,
 66,
 66,
 94,
 78,
 82,
 69,
 63,
 66,
 69,
 93,
 69,
 104,
 68,
 63,
 81,
 81,
 155,
 71,
 93,
 71,
 66,
 66,
 70,
 74,
 124,
 85,
 171,
 130,
 102,
 95,
 184,
 88,
 99,
 79,
 65,
 91,
 99,
 138,
 85,
 73,
 68,
 74,
 82,
 70,
 96,
 82,
 86,
 85,
 8

In [28]:
[len(pred) for pred in new_preds]

[133,
 82,
 78,
 65,
 82,
 74,
 70,
 77,
 70,
 66,
 90,
 73,
 80,
 65,
 63,
 63,
 95,
 91,
 99,
 77,
 88,
 112,
 101,
 64,
 74,
 63,
 81,
 107,
 81,
 90,
 72,
 64,
 78,
 74,
 66,
 89,
 65,
 64,
 83,
 84,
 78,
 70,
 67,
 82,
 92,
 109,
 69,
 103,
 129,
 115,
 77,
 92,
 78,
 89,
 65,
 102,
 85,
 77,
 71,
 70,
 63,
 75,
 83,
 79,
 66,
 103,
 64,
 82,
 74,
 66,
 84,
 65,
 69,
 83,
 64,
 85,
 91,
 193,
 108,
 78,
 66,
 64,
 63,
 97,
 66,
 78,
 80,
 85,
 69,
 93,
 124,
 71,
 64,
 99,
 116,
 86,
 128,
 63,
 92,
 79,
 75,
 65,
 76,
 101,
 63,
 146,
 77,
 68,
 63,
 165,
 68,
 66,
 70,
 102,
 80,
 64,
 0,
 91,
 66,
 83,
 68,
 63,
 64,
 93,
 63,
 113,
 122,
 63,
 84,
 91,
 69,
 89,
 74,
 79,
 110,
 79,
 104,
 81,
 72,
 103,
 105,
 103,
 75,
 68,
 65,
 66,
 66,
 94,
 78,
 82,
 69,
 63,
 66,
 69,
 93,
 69,
 104,
 68,
 63,
 81,
 81,
 155,
 71,
 93,
 71,
 66,
 66,
 70,
 74,
 124,
 85,
 171,
 130,
 102,
 95,
 184,
 88,
 99,
 79,
 65,
 91,
 99,
 138,
 85,
 73,
 68,
 74,
 82,
 70,
 96,
 82,
 86,
 85,
 8

In [29]:
clean_golds, clean_preds = [], []
for g, p in zip(golds, new_preds):
    # se o modelo gerar menos/more tags que tokens, ajustamos:
    if len(p) < len(g):
        print('limpou preds')
        p = p + ["O"] * (len(g) - len(p))          # completa com O
    elif len(p) > len(g):
        print('limpou golds')
        p = p[:len(g)]                             # descarta excedente
    clean_golds.append(g)
    clean_preds.append(p)

limpou golds
limpou preds
limpou golds
limpou preds
limpou golds
limpou preds
limpou preds
limpou preds


In [30]:
from seqeval.metrics import classification_report, f1_score, precision_score, recall_score


In [31]:
print(classification_report(clean_golds, clean_preds))   # por classe e micro/macro

# --- 3.  métricas resumidas ---------------------------------------------------
print("F1-macro : ", f1_score(clean_golds, clean_preds, average="macro"))
print("Precisão :", precision_score(clean_golds, clean_preds, average="macro"))
print("Revocação:", recall_score(clean_golds, clean_preds, average="macro"))

                             precision    recall  f1-score   support

       ambienteSedimentacao       0.05      0.14      0.08        21
            baciaSedimentar       0.54      0.58      0.56        90
                  bentonico       0.00      0.00      0.00         1
           campoPetrolifero       0.00      0.00      0.00         1
constituinteRochaSedimentar       0.28      0.85      0.42        13
   contextoGeologicoDeBacia       0.08      0.05      0.07        75
            elementoQuimico       0.00      0.00      0.00         5
                        eon       0.58      0.65      0.61        23
                      epoca       0.49      0.40      0.44        91
                        era       0.60      0.69      0.64        35
              estratigrafia       0.00      0.00      0.00        27
         estruturaGeologica       0.00      0.00      0.00         7
        estruturaSedimentar       0.00      0.00      0.00         8
                    fosseis      

c:\Users\user\Documents\mestrado\ner_splits\ner_splits\Lib\site-packages\seqeval\metrics\v1.py:57: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
c:\Users\user\Documents\mestrado\ner_splits\ner_splits\Lib\site-packages\seqeval\metrics\v1.py:57: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
